# DIMER Notebook: Semantic Search and Reranking with Qwen3

**Notebook profile:** `MULTI-CAPABILITY`  
**Pedagogical mode:** `WORKSHOP`  
**DIMER Notebook Specification:** `2.1`  
**Standalone:** yes  
**Default workflow:** frozen inference only — no fine-tuning or adapter creation

This notebook composes two live DIMER models into a measurable two-stage search system:

1. **Qwen3-Embedding-0.6B** retrieves a broad candidate set with reusable dense vectors.
2. **Qwen3-Reranker-0.6B** spends more compute on a small shortlist and reorders it with a cross-encoder relevance score.

### Learning objectives

By the end you should be able to:

- distinguish bi-encoder retrieval from cross-encoder reranking;
- precompute and reuse document embeddings;
- retrieve candidates with cosine similarity;
- rerank only the top-*k* candidates;
- measure the retriever and the composed system separately;
- explain why a reranker cannot recover a document omitted by first-stage retrieval;
- inspect cases where reranking helps, hurts, preserves, or cannot recover the gold result; and
- export machine-readable rankings, metrics, embeddings, and provenance.

> **Evidence boundary.** The default data is a small Banking77-derived tutorial retrieval task. Results from this notebook are sample/tutorial evidence, not a production benchmark.

## How to use this notebook

**Who it is for.** Learners who can run Python cells in Colab/Jupyter and are new to embeddings, semantic search, or reranking.

**Runtime.** A CUDA GPU such as a Tesla T4 is recommended; CPU execution is possible but reranking is substantially slower.

**How to run it.**
1. Select the documented runtime/accelerator.
2. Choose **Run all** for the canonical path; the defaults are the reference settings.
3. Read the explanation around each learning stage while the cells execute.
4. Cells marked **Infrastructure** handle setup, model acquisition, provenance, or orchestration. You may run those cells without understanding their implementation.

### Task at a glance

`query → embedding retriever → top-k candidates → cross-encoder reranker → ranked results`

### Roadmap

1. Understand the task, inputs, outputs, and evaluation boundary.
2. Inspect and validate the built-in sample.
3. Establish the simple reference/baseline where applicable.
4. Run the model or model comparison.
5. Inspect errors, disagreements, robustness, or resource tradeoffs.
6. Try one controlled change and explain what changed.
7. Write an evidence-based conclusion, then optionally try your own compatible data.

### What successful execution looks like

You should finish with validated sample/input evidence, the notebook's principal baseline/reference result, model outputs and comparison metrics, at least one diagnostic or qualitative view, and machine-readable results/provenance where the capability supports them. Exact numeric values may vary slightly with the supported runtime; interpret the pattern and the stated metric semantics rather than treating one number as universal.


## 1. How two-stage retrieval works

A dense **embedding retriever** encodes each document independently and can therefore precompute document vectors:

`document → vector`

At query time:

`query → vector → cosine similarity against the index → top-k candidates`

A **cross-encoder reranker** instead evaluates the query and one candidate jointly:

`query + candidate → relevance score`

That richer interaction is more expensive because it requires a model forward pass for every query–candidate pair. A practical search system therefore uses the retriever to reduce the corpus to a shortlist, then applies the reranker only to that shortlist.

The central constraint in this notebook is:

> **The reranker can reorder only the documents it receives. If the relevant document is absent from the first-stage shortlist, it cannot recover it.**

## 2. Configuration

The default values form the canonical `Run all` path. The optional K-sweep and BYOD branches are disabled by default.

In [ ]:
USE_BYOD = False  # @param {type:"boolean"}
DOCUMENTS_PATH = ""  # @param {type:"string"}
QUERIES_PATH = ""  # @param {type:"string"}

SAMPLE_SEED = 42  # @param {type:"integer"}
RERANK_K = 6  # @param {type:"integer"}

EMBEDDING_INSTRUCTION = "Given a customer support message, retrieve the banking intent it expresses"
RERANK_INSTRUCTION = "Given a customer support message, judge whether the document names the banking intent it expresses"

RUN_K_SWEEP = False  # @param {type:"boolean"}
K_SWEEP_VALUES = [3, 6, 10]

OUTPUT_DIR = "outputs/qwen3_semantic_search"

if not 2 <= RERANK_K <= 32:
    raise ValueError("RERANK_K must be in 2..32 for this tutorial.")
if RUN_K_SWEEP and any((not isinstance(k, int) or not 2 <= k <= 32) for k in K_SWEEP_VALUES):
    raise ValueError("Every K_SWEEP_VALUES entry must be an integer in 2..32.")

print({
    "USE_BYOD": USE_BYOD,
    "SAMPLE_SEED": SAMPLE_SEED,
    "RERANK_K": RERANK_K,
    "RUN_K_SWEEP": RUN_K_SWEEP,
    "OUTPUT_DIR": OUTPUT_DIR,
})

## 3. Runtime and reproducibility

The two live DIMER carriers share the same tested dependency set. This notebook uses those exact principal-library pins.

The installation cell is ordinary Python rather than notebook shell magic. On a hosted runtime that already has incompatible packages imported before this notebook starts, the cell fails explicitly instead of silently mixing versions; start a fresh runtime in that case.

**Recommended runtime:** CUDA GPU such as a Tesla T4. CPU works, but cross-encoder reranking is substantially slower.

> **Infrastructure.** This section supports reproducibility and execution. Run the associated setup code as written; understanding its implementation is not a learning objective for this notebook.

In [ ]:
import importlib.metadata as importlib_metadata
import subprocess
import sys

PINS = {
    "torch": "2.14.0",
    "torchvision": "0.29.0",
    "torchaudio": "2.11.0",
    "transformers": "4.57.6",
    "huggingface-hub": "0.36.2",
    "safetensors": "0.8.0",
    "numpy": "2.5.3",
}

def installed_version(dist_name):
    """The public version (PEP 440 without a local label such as +cu126), as pip compares `==` pins."""
    try:
        return importlib_metadata.version(dist_name).split("+", 1)[0]
    except importlib_metadata.PackageNotFoundError:
        return None

before = {name: installed_version(name) for name in PINS}
need_install = [f"{name}=={version}" for name, version in PINS.items() if before[name] != version]

if need_install:
    print("Installing pinned runtime:", need_install)
    completed = subprocess.run(
        [sys.executable, "-m", "pip", "install", "--quiet", *need_install],
        check=False, text=True, capture_output=True,
    )
    if completed.returncode != 0:
        raise RuntimeError(
            f"pip install failed with exit code {completed.returncode}.\n"
            f"--- stderr (tail) ---\n{completed.stderr[-4000:]}\n"
            f"--- stdout (tail) ---\n{completed.stdout[-2000:]}"
        )

after = {name: installed_version(name) for name in PINS}
bad = {name: (after[name], expected) for name, expected in PINS.items() if after[name] != expected}
if bad:
    raise RuntimeError(f"Pinned installation did not converge: {bad}")

# Fail closed if the host pre-imported a now-replaced principal module.
stale = []
for module_name, dist_name in [("torch", "torch"), ("transformers", "transformers"), ("numpy", "numpy")]:
    module = sys.modules.get(module_name)
    if module is not None:
        runtime_version = getattr(module, "__version__", None)
        disk_version = after[dist_name]
        if runtime_version and not str(runtime_version).startswith(str(disk_version)):
            stale.append((module_name, runtime_version, disk_version))
if stale:
    raise RuntimeError(
        "This hosted kernel pre-imported packages that were replaced by the pinned install. "
        f"Start a fresh runtime and run all again. Stale modules: {stale}"
    )

import numpy as np
import torch
import transformers
import huggingface_hub
import safetensors

DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"
DTYPE = torch.bfloat16 if DEVICE.startswith("cuda") else torch.float32

RUNTIME = {
    "python": sys.version.split()[0],
    "torch": torch.__version__,
    "transformers": transformers.__version__,
    "huggingface_hub": huggingface_hub.__version__,
    "numpy": np.__version__,
    "device": DEVICE,
    "dtype": str(DTYPE),
    "cuda_available": torch.cuda.is_available(),
}
print(RUNTIME)

## 4. Immutable model provenance and snapshot verification

The notebook embeds the same immutable snapshot manifests used by the live DIMER carriers.

For each model it:

1. writes the trusted manifest locally;
2. downloads only manifest-listed files from the exact immutable Hugging Face revision;
3. checks byte size and SHA-256 for every listed file; and
4. loads only from the verified local snapshot with `trust_remote_code=False`.

A verification failure is terminal. The notebook does not fall back to another checkpoint or a mutable branch.

> **Infrastructure.** This section supports reproducibility and execution. Run the associated setup code as written; understanding its implementation is not a learning objective for this notebook.

In [ ]:
import hashlib
import json
from pathlib import Path
from huggingface_hub import hf_hub_download

EMBED_MANIFEST = json.loads(r"""{
  "format": "dimer_hf_snapshot",
  "formatVersion": 1,
  "modelKey": "qwen3-embedding-0.6b",
  "modelId": "Qwen/Qwen3-Embedding-0.6B",
  "revision": "97b0c614be4d77ee51c0cef4e5f07c00f9eb65b3",
  "files": [
    {
      "path": "1_Pooling/config.json",
      "bytes": 313,
      "sha256": "37bf193fa101f19101bfad9c31d3eb0f786e247b7b1e5cb7f007d730eed1ddbd"
    },
    {
      "path": "README.md",
      "bytes": 17237,
      "sha256": "c34d9b7e5a267ad3fdd13227a253686bc90844ff4744a2a6a86c7c905e3d06f3"
    },
    {
      "path": "config.json",
      "bytes": 727,
      "sha256": "b5bf1f51fc45be473a54718cef92448d90a1be001bf9b9a44b8c7f10a19feaa9"
    },
    {
      "path": "config_sentence_transformers.json",
      "bytes": 215,
      "sha256": "10667c72ddb772627bf1780cb7f86af8e2ae0032b8c243c731172064105c6961"
    },
    {
      "path": "generation_config.json",
      "bytes": 117,
      "sha256": "28396d421a2108acce96383f6a7de78008f7f1b17f807958f3c14c51dbfb65fb"
    },
    {
      "path": "merges.txt",
      "bytes": 1671853,
      "sha256": "8831e4f1a044471340f7c0a83d7bd71306a5b867e95fd870f74d0c5308a904d5"
    },
    {
      "path": "model.safetensors",
      "bytes": 1191586416,
      "sha256": "0437e45c94563b09e13cb7a64478fc406947a93cb34a7e05870fc8dcd48e23fd"
    },
    {
      "path": "modules.json",
      "bytes": 349,
      "sha256": "84e40c8e006c9b1d6c122e02cba9b02458120b5fb0c87b746c41e0207cf642cf"
    },
    {
      "path": "tokenizer.json",
      "bytes": 11423705,
      "sha256": "def76fb086971c7867b829c23a26261e38d9d74e02139253b38aeb9df8b4b50a"
    },
    {
      "path": "tokenizer_config.json",
      "bytes": 9706,
      "sha256": "253153d0738ceb4c668d2eff957714dd2bea0b56de772a9fdccd96cbf517e6a0"
    },
    {
      "path": "vocab.json",
      "bytes": 2776833,
      "sha256": "ca10d7e9fb3ed18575dd1e277a2579c16d108e32f27439684afa0e10b1440910"
    }
  ],
  "totalBytes": 1207487471
}""")
RERANK_MANIFEST = json.loads(r"""{
  "format": "dimer_hf_snapshot",
  "formatVersion": 1,
  "modelKey": "qwen3-reranker-0.6b",
  "modelId": "Qwen/Qwen3-Reranker-0.6B",
  "revision": "e61197ed45024b0ed8a2d74b80b4d909f1255473",
  "files": [
    {
      "path": "1_LogitScore/config.json",
      "bytes": 57,
      "sha256": "73e3156450564d8a98b7e47bcf5aace0f29600828b51937da545571e84db3ff3"
    },
    {
      "path": "README.md",
      "bytes": 14742,
      "sha256": "5bba8c734f6dd3ae48317b4139317e45a7fce48fc55e15670b23a0dd15492ab6"
    },
    {
      "path": "chat_template.jinja",
      "bytes": 741,
      "sha256": "6f682162495ec5b39fd9005c01b6aa2a74669379fe967039f1e2cbbe8752369d"
    },
    {
      "path": "config.json",
      "bytes": 727,
      "sha256": "d479c427a9ca5295218063d4f9aca4f297ab4ac27487cca7af42c84643d51ef0"
    },
    {
      "path": "config_sentence_transformers.json",
      "bytes": 325,
      "sha256": "6a153d6696f78fd588c1c728967f0b773ea869d3c6028f151ce71ebe49140762"
    },
    {
      "path": "generation_config.json",
      "bytes": 214,
      "sha256": "81051cd3f6e77013827148d0b8a6ead93f8ac390d5ab805f849199f0af6a08db"
    },
    {
      "path": "merges.txt",
      "bytes": 1671853,
      "sha256": "8831e4f1a044471340f7c0a83d7bd71306a5b867e95fd870f74d0c5308a904d5"
    },
    {
      "path": "model.safetensors",
      "bytes": 1191588280,
      "sha256": "27cd75a405b9c1b46b59abfd88aaa209e6fed2a1972cde9b70e7659537c5e65b"
    },
    {
      "path": "modules.json",
      "bytes": 280,
      "sha256": "6f13b6b4a89e577b591b2077bca40c67c26541a6740a8809267cb474f90806a9"
    },
    {
      "path": "sentence_bert_config.json",
      "bytes": 362,
      "sha256": "3234ebd224d492cbe8d55d5ec80a3f408451c4db3005bafb64fe1c51c763e01e"
    },
    {
      "path": "tokenizer.json",
      "bytes": 11422654,
      "sha256": "aeb13307a71acd8fe81861d94ad54ab689df773318809eed3cbe794b4492dae4"
    },
    {
      "path": "tokenizer_config.json",
      "bytes": 9706,
      "sha256": "253153d0738ceb4c668d2eff957714dd2bea0b56de772a9fdccd96cbf517e6a0"
    },
    {
      "path": "vocab.json",
      "bytes": 2776833,
      "sha256": "ca10d7e9fb3ed18575dd1e277a2579c16d108e32f27439684afa0e10b1440910"
    }
  ],
  "totalBytes": 1207486774
}""")

EMBED_DIR = Path("weights/qwen3-embedding-0.6b")
RERANK_DIR = Path("weights/qwen3-reranker-0.6b")
MANIFEST_NAME = "dimer-base-manifest.json"

def sha256_file(path):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for chunk in iter(lambda: handle.read(1 << 20), b""):
            digest.update(chunk)
    return digest.hexdigest()

def write_manifest(root, manifest):
    root.mkdir(parents=True, exist_ok=True)
    path = root / MANIFEST_NAME
    path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")
    return path

def stage_and_verify_snapshot(root, manifest):
    write_manifest(root, manifest)
    model_id = manifest["modelId"]
    revision = manifest["revision"]

    fetched = []
    for entry in manifest["files"]:
        target = root / entry["path"]
        if not target.is_file():
            target.parent.mkdir(parents=True, exist_ok=True)
            hf_hub_download(
                repo_id=model_id,
                filename=entry["path"],
                revision=revision,
                local_dir=str(root),
            )
            fetched.append(entry["path"])

    for entry in manifest["files"]:
        path = root / entry["path"]
        if not path.is_file():
            raise FileNotFoundError(f"Snapshot file missing after staging: {path}")
        actual_size = path.stat().st_size
        if actual_size != entry["bytes"]:
            raise ValueError(
                f"{model_id} {entry['path']} size {actual_size} != {entry['bytes']}"
            )
        actual_sha = sha256_file(path)
        if actual_sha != entry["sha256"]:
            raise ValueError(
                f"{model_id} {entry['path']} sha256 {actual_sha} != {entry['sha256']}"
            )
    return {
        "model_id": model_id,
        "revision": revision,
        "files": len(manifest["files"]),
        "bytes": manifest["totalBytes"],
        "fetched": fetched,
    }

embed_snapshot = stage_and_verify_snapshot(EMBED_DIR, EMBED_MANIFEST)
rerank_snapshot = stage_and_verify_snapshot(RERANK_DIR, RERANK_MANIFEST)

print("Embedding snapshot:", embed_snapshot)
print("Reranker snapshot:", rerank_snapshot)

## 5. Dataset provenance

The default sample uses **Banking77** (Casanueva et al., 2020; CC BY 4.0), fetched from a pinned commit of the PolyAI task-specific datasets repository.

The source contains 13,083 customer-support messages labelled with 77 fine-grained banking intents. In this notebook:

- each intent identifier becomes a short candidate **document** such as `cash withdrawal`;
- the corpus therefore contains exactly **77 candidate documents**;
- the evaluation set contains **154 messages from Banking77's official test partition** — exactly two queries per intent;
- there is no training, adaptation, validation-based selection, or model selection in this notebook.

This is an intentionally small and interpretable retrieval surrogate, not a simulation of large enterprise document search. Pretraining overlap with Banking77 cannot be ruled out.

> **Infrastructure.** This section supports reproducibility and execution. Run the associated setup code as written; understanding its implementation is not a learning objective for this notebook.

In [ ]:
import csv
import io
import random
import urllib.request
from collections import defaultdict

CORPUS_NAME = "Banking77"
CORPUS_LICENSE = "CC BY 4.0 (Casanueva et al. 2020; PolyAI-LDN/task-specific-datasets)"
CORPUS_RELEASE = "PolyAI-LDN/task-specific-datasets @ 57ec275d8078af65b7731c2a98be812d844a6d6b"
CORPUS_BASE_URL = (
    "https://raw.githubusercontent.com/PolyAI-LDN/task-specific-datasets/"
    "57ec275d8078af65b7731c2a98be812d844a6d6b/banking_data/"
)
CORPUS_FILES = {
    "train": ("train.csv", 839_073, "b06e26ac675513959a63135f11b94ea7786ed02da65db93a5650d8838cbc664b"),
    "test": ("test.csv", 239_961, "d12d6e3bc4c3103966ae786dc435913c0c563dfa328f5a3646d0e62cfeeb474d"),
}
CORPUS_ROWS = {"train": 10_003, "test": 3_080}
CORPUS_INTENTS = 77
CORPUS_CACHE = Path("weights/banking77")
MAX_TEXT_CHARS = 100_000

def sha256_bytes(data):
    return hashlib.sha256(data).hexdigest()

def fetch_corpus_file(split):
    name, expected_size, expected_sha = CORPUS_FILES[split]
    CORPUS_CACHE.mkdir(parents=True, exist_ok=True)
    path = CORPUS_CACHE / name

    data = path.read_bytes() if path.is_file() else b""
    if len(data) != expected_size or sha256_bytes(data) != expected_sha:
        with urllib.request.urlopen(CORPUS_BASE_URL + name, timeout=120) as response:
            data = response.read()
        if len(data) != expected_size or sha256_bytes(data) != expected_sha:
            raise ValueError(f"{name}: pinned size/digest verification failed")
        path.write_bytes(data)
    return data

def read_corpus_csv(data, split):
    rows = list(csv.DictReader(io.StringIO(data.decode("utf-8"))))
    if len(rows) != CORPUS_ROWS[split]:
        raise ValueError(f"{split}: {len(rows)} rows != expected {CORPUS_ROWS[split]}")
    if not rows or not {"text", "category"}.issubset(rows[0]):
        raise ValueError(f"{split}: expected columns text and category")
    cleaned = []
    for i, row in enumerate(rows):
        text = row["text"].strip()
        intent = row["category"].strip()
        if not text or not intent:
            continue
        if len(text) > MAX_TEXT_CHARS:
            continue
        cleaned.append({"source_id": f"{split}-{i:05d}", "text": text, "intent": intent})
    intents = {r["intent"] for r in cleaned}
    if len(intents) != CORPUS_INTENTS:
        raise ValueError(f"{split}: {len(intents)} intents != expected {CORPUS_INTENTS}")
    return cleaned

train_rows = read_corpus_csv(fetch_corpus_file("train"), "train")
test_rows = read_corpus_csv(fetch_corpus_file("test"), "test")

print({
    "corpus": CORPUS_NAME,
    "release": CORPUS_RELEASE,
    "license": CORPUS_LICENSE,
    "train_rows": len(train_rows),
    "test_rows": len(test_rows),
    "intents": len({r["intent"] for r in train_rows}),
})

## 6. Build and validate the retrieval problem

Every intent becomes one document. The evaluation queries come only from the official test split.

Validation happens before either model runs:

- 77 unique intent documents;
- unique document IDs and unique normalized document texts;
- 154 unique held-out queries;
- exactly two queries per intent;
- every gold document exists in the corpus;
- no empty or over-limit text.

In [ ]:
def intent_phrase(intent):
    return " ".join(intent.strip().split("_"))

intents = sorted({r["intent"] for r in train_rows})
documents = [
    {
        "doc_id": f"intent_{i:03d}",
        "text": intent_phrase(intent),
        "intent": intent,
    }
    for i, intent in enumerate(intents)
]
intent_to_doc = {d["intent"]: d["doc_id"] for d in documents}
doc_id_to_index = {d["doc_id"]: i for i, d in enumerate(documents)}

by_intent = defaultdict(list)
for row in test_rows:
    by_intent[row["intent"]].append(row)

rng = random.Random(SAMPLE_SEED)
queries = []
for intent in intents:
    pool = list(by_intent[intent])
    rng.shuffle(pool)
    if len(pool) < 2:
        raise ValueError(f"Intent {intent!r} has fewer than two test examples")
    for source in pool[:2]:
        queries.append({
            "query_id": source["source_id"],
            "query": source["text"],
            "gold_doc_id": intent_to_doc[intent],
            "intent": intent,
        })
rng.shuffle(queries)

def validate_problem(documents, queries):
    if len(documents) != 77:
        raise ValueError(f"Expected 77 documents, got {len(documents)}")
    if len(queries) != 154:
        raise ValueError(f"Expected 154 queries, got {len(queries)}")
    doc_ids = [d["doc_id"] for d in documents]
    doc_texts = [d["text"].strip().lower() for d in documents]
    if len(set(doc_ids)) != len(doc_ids):
        raise ValueError("Duplicate doc_id")
    if len(set(doc_texts)) != len(doc_texts):
        raise ValueError("Duplicate normalized document text")
    query_ids = [q["query_id"] for q in queries]
    if len(set(query_ids)) != len(query_ids):
        raise ValueError("Duplicate query_id")
    counts = defaultdict(int)
    for d in documents:
        if not d["text"].strip() or len(d["text"]) > MAX_TEXT_CHARS:
            raise ValueError(f"Invalid document {d['doc_id']}")
    for q in queries:
        if not q["query"].strip() or len(q["query"]) > MAX_TEXT_CHARS:
            raise ValueError(f"Invalid query {q['query_id']}")
        if q["gold_doc_id"] not in doc_id_to_index:
            raise ValueError(f"Missing gold document for {q['query_id']}")
        counts[q["intent"]] += 1
    if set(counts.values()) != {2} or len(counts) != 77:
        raise ValueError("Evaluation set must contain exactly two queries per intent")
    payload = json.dumps(
        {"documents": documents, "queries": queries},
        ensure_ascii=False,
        sort_keys=True,
        separators=(",", ":"),
    ).encode("utf-8")
    return {
        "documents": len(documents),
        "queries": len(queries),
        "queries_per_intent": 2,
        "digest": hashlib.sha256(payload).hexdigest(),
    }

problem_manifest = validate_problem(documents, queries)
print(problem_manifest)
print("Example documents:", documents[:5])
print("Example query:", queries[0])

## 7. Non-neural baselines

Before using either model, we establish two reference points:

- **Random floor:** the expected performance of a uniformly random ordering of 77 documents.
- **Lexical baseline:** Jaccard overlap between lower-cased alphanumeric query/document tokens.

The lexical baseline is intentionally simple. It provides a no-model comparison without introducing another retrieval package.

> **Before you run it:** predict whether this simple reference will be easy or difficult for the learned model(s) to beat. After the result appears, record the baseline value before looking at the more complex model comparison.

In [ ]:
import math
import re
import statistics

TOKEN_RE = re.compile(r"[a-z0-9]+")

def tokens(text):
    return set(TOKEN_RE.findall(text.lower()))

def jaccard(a, b):
    x, y = tokens(a), tokens(b)
    if not x or not y:
        return 0.0
    return len(x & y) / len(x | y)

def pessimistic_rank(scores, positive_index):
    target = float(scores[positive_index])
    return 1 + sum(
        1 for i, score in enumerate(scores)
        if i != positive_index and float(score) >= target
    )

def metrics_from_ranks(ranks, n_documents, ks=(1, 3, 6, 10)):
    if not ranks:
        raise ValueError("No ranks to score")
    result = {
        "n_queries": len(ranks),
        "n_documents": n_documents,
        "mrr": sum(1.0 / r for r in ranks) / len(ranks),
        "median_rank": float(statistics.median(ranks)),
    }
    for k in ks:
        result[f"recall@{k}"] = sum(r <= k for r in ranks) / len(ranks)
    return result

def random_floor(n_documents, ks=(1, 3, 6, 10)):
    result = {
        "n_documents": n_documents,
        "mrr": sum(1.0 / r for r in range(1, n_documents + 1)) / n_documents,
    }
    for k in ks:
        result[f"recall@{k}"] = min(k, n_documents) / n_documents
    return result

lexical_ranks = []
lexical_rows = []
for q in queries:
    scores = [jaccard(q["query"], d["text"]) for d in documents]
    gold_index = doc_id_to_index[q["gold_doc_id"]]
    rank = pessimistic_rank(scores, gold_index)
    lexical_ranks.append(rank)
    order = sorted(range(len(scores)), key=lambda i: (-scores[i], i))
    for r, idx in enumerate(order, start=1):
        lexical_rows.append({
            "query_id": q["query_id"],
            "doc_id": documents[idx]["doc_id"],
            "rank": r,
            "score": float(scores[idx]),
            "is_gold": documents[idx]["doc_id"] == q["gold_doc_id"],
        })

random_metrics = random_floor(len(documents))
lexical_metrics = metrics_from_ranks(lexical_ranks, len(documents))
print("Random floor:", random_metrics)
print("Lexical baseline:", lexical_metrics)

## 8. Stage 1 — build the dense document index

Qwen3-Embedding uses:

- left padding;
- last-token pooling;
- 1024-dimensional representations;
- L2 normalization;
- an instruction prefix for **queries only**.

The 77 document vectors are the reusable offline index. Cosine similarity is implemented as a dot product because both query and document vectors are unit normalized.

Cosine similarity is an ordering signal, not a calibrated probability.

In [ ]:
import time
from transformers import AutoModel, AutoTokenizer

EMBED_MODEL_ID = EMBED_MANIFEST["modelId"]
EMBED_MODEL_REVISION = EMBED_MANIFEST["revision"]
EMBED_DIM = 1024
EMBED_MAX_TOKENS = 8192
EMBED_MAX_BATCH = 64

class EmbeddingRuntime:
    def __init__(self, root, device=DEVICE):
        self.device = device
        self.dtype = torch.bfloat16 if device.startswith("cuda") else torch.float32
        self.tokenizer = AutoTokenizer.from_pretrained(
            str(root),
            padding_side="left",
            trust_remote_code=False,
            local_files_only=True,
        )
        self.model = AutoModel.from_pretrained(
            str(root),
            dtype=self.dtype,
            trust_remote_code=False,
            local_files_only=True,
        ).to(device).eval()

    @staticmethod
    def format_query(query, instruction):
        return f"Instruct: {instruction}\nQuery:{query}"

    def embed_batch(self, texts, kind, instruction):
        if kind not in {"query", "document"}:
            raise ValueError("kind must be query or document")
        if not 1 <= len(texts) <= EMBED_MAX_BATCH:
            raise ValueError(f"Embedding batch must contain 1..{EMBED_MAX_BATCH} texts")
        for i, text in enumerate(texts):
            if not isinstance(text, str) or not text.strip():
                raise ValueError(f"texts[{i}] must be a non-empty string")
            if len(text) > MAX_TEXT_CHARS:
                raise ValueError(f"texts[{i}] exceeds {MAX_TEXT_CHARS} characters")

        formatted = [
            self.format_query(text, instruction) if kind == "query" else text
            for text in texts
        ]
        batch = self.tokenizer(
            formatted,
            padding=True,
            truncation=True,
            max_length=EMBED_MAX_TOKENS,
            return_tensors="pt",
        ).to(self.device)

        with torch.inference_mode():
            hidden = self.model(**batch).last_hidden_state
        pooled = hidden[:, -1].float()
        normalized = torch.nn.functional.normalize(pooled, dim=-1)
        counts = batch["attention_mask"].sum(dim=1).tolist()
        return normalized.cpu().numpy().astype(np.float32), [int(x) for x in counts]

    def embed_all(self, texts, kind, instruction):
        vectors, counts = [], []
        for start in range(0, len(texts), EMBED_MAX_BATCH):
            v, c = self.embed_batch(texts[start:start + EMBED_MAX_BATCH], kind, instruction)
            vectors.append(v)
            counts.extend(c)
        return np.vstack(vectors), counts

t0 = time.perf_counter()
embedder = EmbeddingRuntime(EMBED_DIR)

doc_texts = [d["text"] for d in documents]
document_vectors, document_token_counts = embedder.embed_all(
    doc_texts, "document", EMBEDDING_INSTRUCTION
)
index_seconds = time.perf_counter() - t0

if document_vectors.shape != (77, EMBED_DIM):
    raise RuntimeError(f"Document index shape {document_vectors.shape} != (77, {EMBED_DIM})")
if not np.isfinite(document_vectors).all():
    raise RuntimeError("Document embeddings contain non-finite values")
norms = np.linalg.norm(document_vectors, axis=1)
if not np.allclose(norms, 1.0, atol=2e-4):
    raise RuntimeError(f"Document embeddings are not unit norm: range {norms.min()}..{norms.max()}")

print({
    "n_documents": len(documents),
    "embedding_dim": document_vectors.shape[1],
    "index_build_seconds": round(index_seconds, 3),
    "device": DEVICE,
    "dtype": str(DTYPE),
    "norm_range": [float(norms.min()), float(norms.max())],
})

## 9. Retrieve candidates for every held-out query

We embed all 154 queries with the retrieval instruction and rank all 77 documents by cosine similarity.

The output of this stage is the complete first-stage ranking. We retain it separately from the later reranked shortlist so that the two stages can be evaluated independently.

In [ ]:
t0 = time.perf_counter()
query_vectors, query_token_counts = embedder.embed_all(
    [q["query"] for q in queries],
    "query",
    EMBEDDING_INSTRUCTION,
)
query_embed_seconds = time.perf_counter() - t0

t0 = time.perf_counter()
similarities = query_vectors @ document_vectors.T
retrieval_seconds = time.perf_counter() - t0

retrieval_rankings = []
retrieval_rows = []
retriever_gold_ranks = []

for qi, q in enumerate(queries):
    scores = similarities[qi]
    order = np.argsort(-scores, kind="stable")
    if len(set(order.tolist())) != len(documents):
        raise RuntimeError("Retrieval ranking contains duplicate document positions")

    gold_index = doc_id_to_index[q["gold_doc_id"]]
    gold_rank = int(np.where(order == gold_index)[0][0]) + 1
    retriever_gold_ranks.append(gold_rank)

    ranking = []
    for rank, idx in enumerate(order, start=1):
        row = {
            "query_id": q["query_id"],
            "doc_id": documents[int(idx)]["doc_id"],
            "retrieval_rank": rank,
            "retrieval_score": float(scores[int(idx)]),
            "is_gold": documents[int(idx)]["doc_id"] == q["gold_doc_id"],
        }
        ranking.append(row)
        retrieval_rows.append(dict(row))
    retrieval_rankings.append(ranking)

retriever_metrics = metrics_from_ranks(
    retriever_gold_ranks, len(documents), ks=tuple(sorted({1, 3, 6, 10, RERANK_K}))
)

print("Retriever metrics:", retriever_metrics)
print({
    "query_embedding_seconds": round(query_embed_seconds, 3),
    "retrieval_matrix_seconds": round(retrieval_seconds, 6),
    "mean_query_embedding_seconds": round(query_embed_seconds / len(queries), 6),
})

## 10. Build the reranking shortlist

The default shortlist depth is `RERANK_K = 6`.

`shortlist_coverage@K` is the fraction of queries for which the gold document appears anywhere in the first-stage top-*K*. It is the **maximum possible end-to-end recall@1** for any reranker restricted to that shortlist.

If the gold document is missing here, the second stage has no opportunity to put it first.

In [ ]:
shortlists = []
coverage_flags = []

for q, ranking in zip(queries, retrieval_rankings, strict=True):
    topk = ranking[:RERANK_K]
    if len(topk) != RERANK_K:
        raise RuntimeError("Shortlist length mismatch")
    if len({row["doc_id"] for row in topk}) != RERANK_K:
        raise RuntimeError("Shortlist contains duplicate documents")

    present = any(row["is_gold"] for row in topk)
    coverage_flags.append(present)
    shortlists.append({
        "query_id": q["query_id"],
        "query": q["query"],
        "gold_doc_id": q["gold_doc_id"],
        "candidates": topk,
    })

shortlist_coverage = sum(coverage_flags) / len(coverage_flags)
print({
    "rerank_k": RERANK_K,
    "shortlist_coverage@k": shortlist_coverage,
    "covered_queries": sum(coverage_flags),
    "unrecoverable_queries": len(coverage_flags) - sum(coverage_flags),
})

## 11. Prediction exercise

Inspect the first shortlist below before reading the later reranker output.

Consider:

1. Is the gold document already present?
2. Is it currently at rank 1?
3. Which candidate phrases appear semantically close enough that a cross-encoder might reorder them?

The next stage will reveal what the reranker actually does. The exercise is interpretive and does not block `Run all`.

In [ ]:
exercise = shortlists[0]
print("Query:", exercise["query"])
print("Gold:", exercise["gold_doc_id"])
print("Embedding shortlist:")
for row in exercise["candidates"]:
    marker = " <-- GOLD" if row["is_gold"] else ""
    print(f"  {row['retrieval_rank']:>2}. {row['doc_id']:<12} score={row['retrieval_score']:.4f}{marker}")

## 12. Stage 2 — cross-encoder reranking

Qwen3-Reranker scores each query/document pair using the exact live-carrier contract:

- fixed system/user/assistant prompt;
- longest-first truncation within an 8,192-token prompt budget;
- final-position logits for the tokens `no` and `yes`;
- two-way softmax;
- the `yes` share is the returned relevance score.

The score is **not a calibrated probability** and no threshold is applied. It is used only to order candidates supplied for the same query.

To reduce memory pressure, we keep the computed embedding vectors and remove the embedding model before loading the reranker.

In [ ]:
import gc
del embedder
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

from transformers import AutoModelForCausalLM

RERANK_MODEL_ID = RERANK_MANIFEST["modelId"]
RERANK_MODEL_REVISION = RERANK_MANIFEST["revision"]
RERANK_MAX_TOKENS = 8192
RERANK_MAX_PAIRS = 32
YES_TOKEN, NO_TOKEN = "yes", "no"
YES_TOKEN_ID, NO_TOKEN_ID = 9693, 2152

PREFIX = (
    "<|im_start|>system\nJudge whether the Document meets the requirements based on the Query and the "
    'Instruct provided. Note that the answer can only be "yes" or "no".<|im_end|>\n<|im_start|>user\n'
)
SUFFIX = "<|im_end|>\n<|im_start|>assistant\n<think>\n\n</think>\n\n"

class RerankerRuntime:
    def __init__(self, root, device=DEVICE):
        self.device = device
        self.dtype = torch.bfloat16 if device.startswith("cuda") else torch.float32
        self.tokenizer = AutoTokenizer.from_pretrained(
            str(root),
            padding_side="left",
            trust_remote_code=False,
            local_files_only=True,
        )
        ids = (
            self.tokenizer.convert_tokens_to_ids(YES_TOKEN),
            self.tokenizer.convert_tokens_to_ids(NO_TOKEN),
        )
        if ids != (YES_TOKEN_ID, NO_TOKEN_ID):
            raise RuntimeError(f"Tokenizer maps yes/no to {ids}, expected {(YES_TOKEN_ID, NO_TOKEN_ID)}")

        self.model = AutoModelForCausalLM.from_pretrained(
            str(root),
            dtype=self.dtype,
            trust_remote_code=False,
            local_files_only=True,
        ).to(device).eval()

        self.prefix_ids = self.tokenizer.encode(PREFIX, add_special_tokens=False)
        self.suffix_ids = self.tokenizer.encode(SUFFIX, add_special_tokens=False)
        self.body_budget = RERANK_MAX_TOKENS - len(self.prefix_ids) - len(self.suffix_ids)

    @staticmethod
    def format_pair(query, document, instruction):
        return f"<Instruct>: {instruction}\n<Query>: {query}\n<Document>: {document}"

    def score_batch(self, pairs, instruction):
        if not 1 <= len(pairs) <= RERANK_MAX_PAIRS:
            raise ValueError(f"Reranker batch must contain 1..{RERANK_MAX_PAIRS} pairs")
        bodies = []
        for i, (query, document) in enumerate(pairs):
            if not isinstance(query, str) or not query.strip():
                raise ValueError(f"pairs[{i}] query must be a non-empty string")
            if not isinstance(document, str) or not document.strip():
                raise ValueError(f"pairs[{i}] document must be a non-empty string")
            if len(query) > MAX_TEXT_CHARS or len(document) > MAX_TEXT_CHARS:
                raise ValueError(f"pairs[{i}] exceeds the character ceiling")
            bodies.append(self.format_pair(query, document, instruction))

        enc = self.tokenizer(
            bodies,
            padding=False,
            truncation="longest_first",
            return_attention_mask=False,
            max_length=self.body_budget,
        )
        enc["input_ids"] = [
            self.prefix_ids + row + self.suffix_ids
            for row in enc["input_ids"]
        ]
        batch = self.tokenizer.pad(enc, padding=True, return_tensors="pt").to(self.device)

        with torch.inference_mode():
            last = self.model(**batch).logits[:, -1, :]
        logits = torch.stack(
            [last[:, NO_TOKEN_ID], last[:, YES_TOKEN_ID]], dim=1
        ).float()
        probs = torch.softmax(logits, dim=1)[:, 1]
        counts = batch["attention_mask"].sum(dim=1).tolist()
        return probs.cpu().numpy().astype(np.float64), [int(x) for x in counts]

    def score_all(self, pairs, instruction):
        scores, counts = [], []
        for start in range(0, len(pairs), RERANK_MAX_PAIRS):
            s, c = self.score_batch(pairs[start:start + RERANK_MAX_PAIRS], instruction)
            scores.extend(float(x) for x in s)
            counts.extend(c)
        return scores, counts

reranker = RerankerRuntime(RERANK_DIR)
print({
    "model_id": RERANK_MODEL_ID,
    "revision": RERANK_MODEL_REVISION,
    "device": DEVICE,
    "score_kind": "relevance score, not a calibrated probability",
})

## 13. Rerank every top-*K* shortlist

For each query we score exactly the documents retrieved into its shortlist, preserve the original retrieval rank/score, then assign a new rank by descending reranker score.

No candidate is added after the first-stage retrieval boundary.

In [ ]:
pairs = []
pair_meta = []

for shortlist in shortlists:
    for candidate in shortlist["candidates"]:
        doc = documents[doc_id_to_index[candidate["doc_id"]]]
        pairs.append((shortlist["query"], doc["text"]))
        pair_meta.append({
            "query_id": shortlist["query_id"],
            "doc_id": candidate["doc_id"],
            "retrieval_rank": candidate["retrieval_rank"],
            "retrieval_score": candidate["retrieval_score"],
            "is_gold": candidate["is_gold"],
        })

t0 = time.perf_counter()
reranker_scores, reranker_token_counts = reranker.score_all(pairs, RERANK_INSTRUCTION)
rerank_seconds = time.perf_counter() - t0

if len(reranker_scores) != len(queries) * RERANK_K:
    raise RuntimeError("Reranker result count does not match query × shortlist depth")
if not all(math.isfinite(s) and 0.0 <= s <= 1.0 for s in reranker_scores):
    raise RuntimeError("Reranker returned non-finite or out-of-range scores")

reranked_by_query = {}
reranked_rows = []
cursor = 0
for shortlist in shortlists:
    rows = []
    for _ in range(RERANK_K):
        meta = dict(pair_meta[cursor])
        meta["reranker_score"] = float(reranker_scores[cursor])
        rows.append(meta)
        cursor += 1
    rows.sort(key=lambda r: (-r["reranker_score"], r["retrieval_rank"]))
    for rank, row in enumerate(rows, start=1):
        row["reranked_rank"] = rank
        reranked_rows.append(dict(row))
    reranked_by_query[shortlist["query_id"]] = rows

print({
    "pairs_scored": len(pairs),
    "rerank_seconds": round(rerank_seconds, 3),
    "mean_seconds_per_pair": round(rerank_seconds / len(pairs), 6),
})

## 14. End-to-end evaluation

We report two different views.

### Full system

Every one of the 154 queries counts. If the gold document is absent from the shortlist:

- end-to-end recall receives no credit;
- reciprocal-rank contribution is `0`.

This prevents reranker evaluation from hiding first-stage retrieval failures.

### Conditional reranker

We also report ranking quality **only for queries where the gold document entered the shortlist**. These conditional metrics isolate what the second stage did with candidates it actually received, but they do not replace full-system metrics.

> **What to notice.** Compare the model result with the stated baseline/reference first. Then inspect the secondary metric or diagnostic that explains *how* the result was achieved; do not infer a universal model ranking from one sample and configuration.

In [ ]:
pipeline_gold_ranks = []
conditional_gold_ranks = []
effects = {"helped": 0, "hurt": 0, "unchanged": 0, "unrecoverable": 0}
effect_by_query = {}

for q, shortlist in zip(queries, shortlists, strict=True):
    qid = q["query_id"]
    before_gold = next((row["retrieval_rank"] for row in shortlist["candidates"] if row["is_gold"]), None)
    after_gold = next((row["reranked_rank"] for row in reranked_by_query[qid] if row["is_gold"]), None)

    if before_gold is None:
        effects["unrecoverable"] += 1
        effect_by_query[qid] = "unrecoverable"
        pipeline_gold_ranks.append(None)
        continue

    conditional_gold_ranks.append(after_gold)
    pipeline_gold_ranks.append(after_gold)

    if after_gold < before_gold:
        category = "helped"
    elif after_gold > before_gold:
        category = "hurt"
    else:
        category = "unchanged"
    effects[category] += 1
    effect_by_query[qid] = category

n_queries = len(queries)
pipeline_metrics = {
    "n_queries": n_queries,
    "rerank_k": RERANK_K,
    "shortlist_coverage@k": shortlist_coverage,
    "recall@1": sum(r == 1 for r in pipeline_gold_ranks) / n_queries,
    "recall@3": sum(r is not None and r <= 3 for r in pipeline_gold_ranks) / n_queries,
    f"recall@{RERANK_K}": sum(r is not None and r <= RERANK_K for r in pipeline_gold_ranks) / n_queries,
    "mrr": sum(0.0 if r is None else 1.0 / r for r in pipeline_gold_ranks) / n_queries,
}

conditional_metrics = metrics_from_ranks(
    conditional_gold_ranks,
    RERANK_K,
    ks=(1, 3, RERANK_K),
)

if pipeline_metrics["recall@1"] > shortlist_coverage + 1e-12:
    raise RuntimeError("Pipeline recall@1 cannot exceed shortlist coverage")

effect_percent = {k: v / n_queries for k, v in effects.items()}

print("Pipeline metrics:", pipeline_metrics)
print("Conditional reranker metrics:", conditional_metrics)
print("Reranking effects:", effects)
print("Reranking effect percentages:", effect_percent)

## 15. Compare the systems

The rows below answer different questions:

- **Random:** what a uniform ordering would achieve in expectation.
- **Lexical:** what simple word overlap achieves without a neural model.
- **Qwen3 Embedding:** quality of the first-stage dense retriever over all 77 documents.
- **Embedding → Reranker:** quality of the composed system when the reranker sees only the top `RERANK_K`.

There is deliberately **no assertion that reranking must improve the result**. If this measured sample gets worse, that is a valid finding to inspect rather than something to tune away on the test set.

In [ ]:
def fmt_pct(x):
    return f"{100*x:6.2f}%"

summary_rows = [
    {
        "system": "Random ranking (expected)",
        "pool": 77,
        "recall@1": random_metrics["recall@1"],
        "recall@3": random_metrics["recall@3"],
        "recall@6": random_metrics["recall@6"],
        "mrr": random_metrics["mrr"],
    },
    {
        "system": "Lexical Jaccard",
        "pool": 77,
        "recall@1": lexical_metrics["recall@1"],
        "recall@3": lexical_metrics["recall@3"],
        "recall@6": lexical_metrics["recall@6"],
        "mrr": lexical_metrics["mrr"],
    },
    {
        "system": "Qwen3 Embedding",
        "pool": 77,
        "recall@1": retriever_metrics["recall@1"],
        "recall@3": retriever_metrics["recall@3"],
        "recall@6": retriever_metrics["recall@6"],
        "mrr": retriever_metrics["mrr"],
    },
    {
        "system": f"Embedding → Reranker top-{RERANK_K}",
        "pool": RERANK_K,
        "recall@1": pipeline_metrics["recall@1"],
        "recall@3": pipeline_metrics["recall@3"],
        "recall@6": pipeline_metrics[f"recall@{RERANK_K}"],
        "mrr": pipeline_metrics["mrr"],
    },
]

print(f"{'System':<34} {'Pool':>4} {'R@1':>9} {'R@3':>9} {'R@6':>9} {'MRR':>8}")
print("-" * 80)
for row in summary_rows:
    print(
        f"{row['system']:<34} {row['pool']:>4} "
        f"{fmt_pct(row['recall@1']):>9} {fmt_pct(row['recall@3']):>9} "
        f"{fmt_pct(row['recall@6']):>9} {row['mrr']:>8.4f}"
    )

## 16. Failure analysis: where reranking helps and where it cannot

The notebook automatically selects one example from each available category:

- **helped** — gold rank improved;
- **hurt** — gold rank worsened;
- **unchanged** — gold rank stayed the same;
- **unrecoverable** — the retriever omitted the gold document from the shortlist.

This is often more informative than an aggregate metric because it exposes the boundary between candidate generation and candidate ordering.

In [ ]:
query_by_id = {q["query_id"]: q for q in queries}
shortlist_by_id = {s["query_id"]: s for s in shortlists}

def show_case(query_id):
    q = query_by_id[query_id]
    category = effect_by_query[query_id]
    print("\n" + "=" * 90)
    print(f"CASE: {category.upper()}")
    print("Query:", q["query"])
    print("Gold:", q["gold_doc_id"])
    print("\nFirst-stage shortlist:")
    for row in shortlist_by_id[query_id]["candidates"]:
        marker = " <-- GOLD" if row["is_gold"] else ""
        print(
            f"  {row['retrieval_rank']:>2}. {row['doc_id']:<12} "
            f"cos={row['retrieval_score']:.4f}{marker}"
        )
    print("\nReranked shortlist:")
    if category == "unrecoverable":
        print("  Gold document was not supplied to the reranker.")
    for row in reranked_by_query[query_id]:
        marker = " <-- GOLD" if row["is_gold"] else ""
        print(
            f"  {row['reranked_rank']:>2}. {row['doc_id']:<12} "
            f"rerank={row['reranker_score']:.4f} "
            f"(retrieval rank {row['retrieval_rank']}){marker}"
        )

for category in ("helped", "hurt", "unchanged", "unrecoverable"):
    match = next((qid for qid, value in effect_by_query.items() if value == category), None)
    if match is not None:
        show_case(match)
    else:
        print(f"\nNo {category!r} case occurred in this deterministic sample.")

## 17. Optional experiment — rerank depth versus compute

A larger shortlist gives the reranker access to more candidates and can increase the ceiling imposed by first-stage recall, but it also requires more cross-encoder pair evaluations.

The optional sweep evaluates `K = 3, 6, 10` by default. It is disabled on the canonical path because it multiplies reranker work and must not be used to tune `RERANK_K` on the same held-out queries reported above.

In [ ]:
k_sweep_results = []

if RUN_K_SWEEP:
    for k in K_SWEEP_VALUES:
        sweep_pairs = []
        sweep_meta = []
        coverage = 0
        for q, ranking in zip(queries, retrieval_rankings, strict=True):
            topk = ranking[:k]
            coverage += any(row["is_gold"] for row in topk)
            for candidate in topk:
                doc = documents[doc_id_to_index[candidate["doc_id"]]]
                sweep_pairs.append((q["query"], doc["text"]))
                sweep_meta.append((q["query_id"], candidate["doc_id"], candidate["is_gold"]))

        t0 = time.perf_counter()
        sweep_scores, _ = reranker.score_all(sweep_pairs, RERANK_INSTRUCTION)
        wall = time.perf_counter() - t0

        grouped = defaultdict(list)
        for meta, score in zip(sweep_meta, sweep_scores, strict=True):
            qid, doc_id, is_gold = meta
            grouped[qid].append((float(score), doc_id, is_gold))

        ranks = []
        for q in queries:
            rows = sorted(grouped[q["query_id"]], key=lambda x: -x[0])
            gold_rank = next((i + 1 for i, row in enumerate(rows) if row[2]), None)
            ranks.append(gold_rank)

        result = {
            "k": k,
            "shortlist_coverage": coverage / len(queries),
            "recall@1": sum(r == 1 for r in ranks) / len(ranks),
            "mrr": sum(0.0 if r is None else 1.0 / r for r in ranks) / len(ranks),
            "pairs": len(sweep_pairs),
            "reranker_seconds": wall,
        }
        k_sweep_results.append(result)

    print(f"{'K':>3} {'coverage':>10} {'R@1':>10} {'MRR':>9} {'pairs':>8} {'seconds':>10}")
    for row in k_sweep_results:
        print(
            f"{row['k']:>3} {fmt_pct(row['shortlist_coverage']):>10} "
            f"{fmt_pct(row['recall@1']):>10} {row['mrr']:>9.4f} "
            f"{row['pairs']:>8} {row['reranker_seconds']:>10.2f}"
        )
else:
    print("K sweep disabled on the canonical Run all path.")

## 18. Export machine-readable results and provenance

The notebook writes:

- `document_embeddings.npz`
- `retrieval_results.csv`
- `reranked_results.csv`
- `metrics.json`
- `provenance.json`

Identifiers are preserved so downstream consumers can map every vector and ranking back to its source document/query.

Measured timings belong only to this execution environment.

> **Infrastructure.** This section supports reproducibility and execution. Run the associated setup code as written; understanding its implementation is not a learning objective for this notebook.

In [ ]:
from datetime import datetime, timezone

out_dir = Path(OUTPUT_DIR)
out_dir.mkdir(parents=True, exist_ok=True)

np.savez_compressed(
    out_dir / "document_embeddings.npz",
    doc_ids=np.array([d["doc_id"] for d in documents], dtype=str),
    embeddings=document_vectors.astype(np.float32),
)

with open(out_dir / "retrieval_results.csv", "w", encoding="utf-8", newline="") as handle:
    fieldnames = ["query_id", "doc_id", "retrieval_rank", "retrieval_score", "is_gold"]
    writer = csv.DictWriter(handle, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(retrieval_rows)

with open(out_dir / "reranked_results.csv", "w", encoding="utf-8", newline="") as handle:
    fieldnames = [
        "query_id", "doc_id", "retrieval_rank", "retrieval_score",
        "reranked_rank", "reranker_score", "is_gold",
    ]
    writer = csv.DictWriter(handle, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(reranked_rows)

metrics_export = {
    "random": random_metrics,
    "lexical": lexical_metrics,
    "retriever": retriever_metrics,
    "shortlist": {
        "k": RERANK_K,
        "coverage": shortlist_coverage,
    },
    "pipeline": pipeline_metrics,
    "conditional_reranker": conditional_metrics,
    "effects": {
        "counts": effects,
        "fractions": effect_percent,
    },
    "timings_seconds": {
        "document_index": index_seconds,
        "query_embedding": query_embed_seconds,
        "retrieval_matrix": retrieval_seconds,
        "reranking": rerank_seconds,
    },
    "k_sweep": k_sweep_results,
}
(out_dir / "metrics.json").write_text(
    json.dumps(metrics_export, indent=2),
    encoding="utf-8",
)

provenance = {
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "notebook_spec": "2.1",
    "profile": "MULTI-CAPABILITY",
    "pedagogical_mode": "WORKSHOP",
    "standalone": True,
    "embedding_model": {
        "id": EMBED_MODEL_ID,
        "revision": EMBED_MODEL_REVISION,
        "model_safetensors_sha256": next(
            x["sha256"] for x in EMBED_MANIFEST["files"] if x["path"] == "model.safetensors"
        ),
        "instruction": EMBEDDING_INSTRUCTION,
    },
    "reranker_model": {
        "id": RERANK_MODEL_ID,
        "revision": RERANK_MODEL_REVISION,
        "model_safetensors_sha256": next(
            x["sha256"] for x in RERANK_MANIFEST["files"] if x["path"] == "model.safetensors"
        ),
        "instruction": RERANK_INSTRUCTION,
        "score_kind": "relevance score, not a calibrated probability",
    },
    "dataset": {
        "name": CORPUS_NAME,
        "license": CORPUS_LICENSE,
        "release": CORPUS_RELEASE,
        "source_files": {
            split: {"file": spec[0], "bytes": spec[1], "sha256": spec[2]}
            for split, spec in CORPUS_FILES.items()
        },
        "sample_seed": SAMPLE_SEED,
        "sample_digest": problem_manifest["digest"],
        "documents": len(documents),
        "queries": len(queries),
    },
    "configuration": {
        "rerank_k": RERANK_K,
        "run_k_sweep": RUN_K_SWEEP,
    },
    "runtime": RUNTIME,
    "timings_seconds": metrics_export["timings_seconds"],
}
(out_dir / "provenance.json").write_text(
    json.dumps(provenance, indent=2),
    encoding="utf-8",
)

print("Wrote:")
for path in sorted(out_dir.iterdir()):
    print(" ", path)

## 19. Bring Your Own Data (optional)

The canonical path above is complete. BYOD is optional and disabled by default.

### Documents CSV

Required columns:

`doc_id,text`

Constraints:

- 2–1,000 documents;
- unique non-empty `doc_id`;
- non-empty text;
- each document ≤ 100,000 characters.

The notebook does **not** chunk long documents automatically. Pre-chunk them into meaningful retrievable units.

### Queries CSV

Required columns:

`query_id,query`

Optional evaluation column:

`gold_doc_id`

If all queries contain a valid `gold_doc_id`, the branch reports retrieval/reranking metrics. Without labels, the evaluation verdict is `not-measurable`, but rankings are still produced.

### Privacy

This standalone workflow processes supplied text inside the selected notebook runtime and does not intentionally transmit it to DIMER services. Do not upload confidential, restricted, sensitive, personal, regulated, or otherwise unauthorized material to a hosted notebook environment.

In [ ]:
byod_result = None

def load_csv_records(path, required):
    p = Path(path)
    if not p.is_file():
        raise FileNotFoundError(f"File not found: {p}")
    with open(p, encoding="utf-8", newline="") as handle:
        rows = list(csv.DictReader(handle))
    if not rows:
        raise ValueError(f"{p} is empty")
    missing = set(required) - set(rows[0])
    if missing:
        raise ValueError(f"{p} is missing columns {sorted(missing)}")
    return rows

if USE_BYOD:
    if not DOCUMENTS_PATH or not QUERIES_PATH:
        raise ValueError("Set DOCUMENTS_PATH and QUERIES_PATH when USE_BYOD=True")

    byod_docs = load_csv_records(DOCUMENTS_PATH, {"doc_id", "text"})
    byod_queries = load_csv_records(QUERIES_PATH, {"query_id", "query"})

    if not 2 <= len(byod_docs) <= 1000:
        raise ValueError("BYOD documents must contain 2..1000 rows")
    if not 1 <= len(byod_queries) <= 500:
        raise ValueError("BYOD queries must contain 1..500 rows")

    doc_ids = [r["doc_id"].strip() for r in byod_docs]
    if any(not x for x in doc_ids) or len(set(doc_ids)) != len(doc_ids):
        raise ValueError("BYOD doc_id values must be unique and non-empty")
    query_ids = [r["query_id"].strip() for r in byod_queries]
    if any(not x for x in query_ids) or len(set(query_ids)) != len(query_ids):
        raise ValueError("BYOD query_id values must be unique and non-empty")

    for i, row in enumerate(byod_docs):
        if not row["text"].strip() or len(row["text"]) > MAX_TEXT_CHARS:
            raise ValueError(f"BYOD document row {i} has invalid text")
    for i, row in enumerate(byod_queries):
        if not row["query"].strip() or len(row["query"]) > MAX_TEXT_CHARS:
            raise ValueError(f"BYOD query row {i} has invalid query")

    labelled = all((row.get("gold_doc_id") or "").strip() for row in byod_queries)
    if labelled:
        unknown = sorted({
            row["gold_doc_id"].strip()
            for row in byod_queries
            if row["gold_doc_id"].strip() not in set(doc_ids)
        })
        if unknown:
            raise ValueError(f"BYOD queries reference unknown gold_doc_id values: {unknown[:10]}")

    # Reload the embedder because the canonical path intentionally released it before reranking.
    byod_embedder = EmbeddingRuntime(EMBED_DIR)
    byod_doc_vectors, _ = byod_embedder.embed_all(
        [r["text"].strip() for r in byod_docs],
        "document",
        EMBEDDING_INSTRUCTION,
    )
    byod_query_vectors, _ = byod_embedder.embed_all(
        [r["query"].strip() for r in byod_queries],
        "query",
        EMBEDDING_INSTRUCTION,
    )
    byod_sim = byod_query_vectors @ byod_doc_vectors.T

    byod_shortlists = []
    byod_retrieval_ranks = []
    for qi, row in enumerate(byod_queries):
        order = np.argsort(-byod_sim[qi], kind="stable")
        topk = order[: min(RERANK_K, len(byod_docs))]
        if labelled:
            gold_idx = doc_ids.index(row["gold_doc_id"].strip())
            byod_retrieval_ranks.append(int(np.where(order == gold_idx)[0][0]) + 1)
        byod_shortlists.append(topk.tolist())

    del byod_embedder
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    byod_pairs = []
    byod_meta = []
    for qi, row in enumerate(byod_queries):
        for idx in byod_shortlists[qi]:
            byod_pairs.append((row["query"].strip(), byod_docs[idx]["text"].strip()))
            byod_meta.append((qi, idx))

    byod_scores, _ = reranker.score_all(byod_pairs, RERANK_INSTRUCTION)

    byod_rows = []
    cursor = 0
    byod_final_ranks = []
    for qi, row in enumerate(byod_queries):
        items = []
        for retrieval_rank, idx in enumerate(byod_shortlists[qi], start=1):
            items.append({
                "query_id": row["query_id"].strip(),
                "doc_id": doc_ids[idx],
                "retrieval_rank": retrieval_rank,
                "retrieval_score": float(byod_sim[qi, idx]),
                "reranker_score": float(byod_scores[cursor]),
                "is_gold": labelled and doc_ids[idx] == row["gold_doc_id"].strip(),
            })
            cursor += 1
        items.sort(key=lambda x: (-x["reranker_score"], x["retrieval_rank"]))
        for reranked_rank, item in enumerate(items, start=1):
            item["reranked_rank"] = reranked_rank
            byod_rows.append(item)
        if labelled:
            byod_final_ranks.append(
                next((x["reranked_rank"] for x in items if x["is_gold"]), None)
            )

    byod_result = {
        "documents": len(byod_docs),
        "queries": len(byod_queries),
        "labelled": labelled,
        "evaluation_verdict": "measured" if labelled else "not-measurable",
        "rows": byod_rows,
    }

    if labelled:
        byod_result["retriever_metrics"] = metrics_from_ranks(
            byod_retrieval_ranks, len(byod_docs), ks=(1, 3, min(RERANK_K, len(byod_docs)))
        )
        byod_result["pipeline_recall@1"] = (
            sum(r == 1 for r in byod_final_ranks) / len(byod_final_ranks)
        )
        byod_result["pipeline_mrr"] = (
            sum(0.0 if r is None else 1.0 / r for r in byod_final_ranks) / len(byod_final_ranks)
        )
    else:
        byod_result["reason"] = "gold_doc_id was not supplied for every query"

    print({
        key: value for key, value in byod_result.items()
        if key != "rows"
    })
else:
    print("BYOD disabled on the canonical Run all path.")

## 20. Interpretation, limitations, and transfer

### Retrieval and reranking solve different problems

Dense retrieval efficiently searches a corpus using reusable document representations. Cross-encoder reranking spends more computation on query–candidate interactions after the search space has already been reduced.

### Reranking does not repair missing candidates

First-stage `recall@K` / shortlist coverage is a hard ceiling for a reranker restricted to that shortlist.

### Scores are not calibrated probabilities

Embedding cosine similarity and Qwen3 reranker relevance scores are ranking signals. Their numeric values should not be interpreted as universal confidence or probability.

### Banking77 is tutorial evidence

This notebook has:

- 77 short intent phrases;
- one gold intent per query;
- 154 deterministic held-out queries.

It does **not** establish performance on long documents, multilingual enterprise corpora, graded relevance, large-scale indexing, or production traffic.

### Pretraining overlap is unknown

This notebook cannot establish that Banking77 or equivalent text was absent from upstream pretraining.

### Real retrieval can have multiple relevant documents

A real information-retrieval evaluation commonly needs binary or graded relevance judgements for several candidates per query. The one-label Banking77 construction is simpler.

### Scaling

Document embeddings can be precomputed. Cross-encoder work scales approximately with:

`number of queries × rerank depth`

That cost structure is the reason rerankers usually sit behind a first-stage retriever.

### Next experiment

A clean follow-on is a separate grounded-generation notebook that consumes these ranked results and adds an instruction-following language model. Keeping generation separate here makes retrieval quality measurable on its own.

## 21. Terminal summary

The final cell checks that all required exports exist and prints the measured results from this execution.

In [ ]:
required_outputs = [
    out_dir / "document_embeddings.npz",
    out_dir / "retrieval_results.csv",
    out_dir / "reranked_results.csv",
    out_dir / "metrics.json",
    out_dir / "provenance.json",
]
missing = [str(path) for path in required_outputs if not path.is_file()]
if missing:
    raise RuntimeError(f"Required outputs missing: {missing}")

print("DIMER Qwen3 Semantic Search + Reranking Workshop")
print("-" * 55)
print(f"Queries evaluated: {len(queries)}")
print(f"Documents indexed: {len(documents)}")
print(f"Rerank depth: {RERANK_K}")
print()
print(f"Lexical recall@1:              {lexical_metrics['recall@1']:.4f}")
print(f"Embedding recall@1:            {retriever_metrics['recall@1']:.4f}")
print(f"Embedding recall@{RERANK_K}:            {retriever_metrics[f'recall@{RERANK_K}']:.4f}")
print(f"Shortlist coverage@{RERANK_K}:          {shortlist_coverage:.4f}")
print(f"Two-stage recall@1:            {pipeline_metrics['recall@1']:.4f}")
print(f"Two-stage MRR:                 {pipeline_metrics['mrr']:.4f}")
print()
print("Reranking effects:")
for key in ("helped", "hurt", "unchanged", "unrecoverable"):
    print(f"  {key:<13}: {effects[key]:>3} ({effect_percent[key]:.1%})")
print()
print(f"Outputs: {out_dir}/")
print("Required output files:", len(required_outputs))

## Try it yourself — one controlled change

Use the same discipline as the main experiment:

**Predict → change one variable → rerun → observe → explain**

Change `RERANK_K` from `6` to `3` or `10`. Before rerunning, predict how a shorter or longer shortlist will affect retrieval coverage, reranking compute, and final ranking quality. Rerun from **Configuration** through **Compare the systems**, then explain whether the result matched your prediction.

Keep this exercise separate from the frozen canonical test result. If you use validation or an optional post-test exercise to explore a setting, do not retroactively present the changed setting as the pre-registered canonical result.


## Self-paced checkpoint

Before reading the sample interpretation, answer in your own words:

1. What was the model or system asked to do?
2. Which baseline/reference tells you whether the model added useful capability?
3. What is the most important failure mode or tradeoff visible in this notebook?
4. What would you need to test before applying the result to a different dataset or operational setting?

<details>
<summary><b>Show a sample interpretation</b></summary>

A reranker can improve ordering only inside the shortlist it receives. If the relevant document is absent after first-stage retrieval, reranking cannot recover it. Compare first-stage recall with end-to-end ranking metrics before attributing a failure to the reranker.

A complete answer should cite the outputs from **your run**, because small numerical differences can occur across supported runtimes.

</details>


## Write an evidence-based conclusion

Use the outputs from your run rather than declaring a universal winner.

1. **State the question.** What capability or comparison did this notebook test?
2. **Report the primary result.** Compare the relevant model/system with its baseline or reference on the held-out or otherwise designated evaluation data.
3. **Add supporting evidence.** Include the most informative secondary metric, error pattern, qualitative diagnostic, or disagreement.
4. **Account for cost or complexity.** Mention runtime, model footprint, extra stages, or adaptation when they materially affect the comparison.
5. **State the limits.** Say what this dataset, split, model revision, and configuration do—and do not—support.

State whether two-stage retrieval improved the held-out ranking relative to the non-neural and embedding-only references. Report the principal retrieval metric(s), note whether the relevant item was present in the first-stage shortlist, describe the compute tradeoff introduced by reranking, and limit the conclusion to this Banking77-derived tutorial task.


# Troubleshooting

| What you see | Likely cause | What to do |
|---|---|---|
| Accelerator unavailable or the model is unexpectedly slow | The runtime does not match the documented resource envelope | Select the documented accelerator/runtime, start a fresh session, and run the notebook top-to-bottom. |
| Package/version or stale-module error after installation | The hosted kernel had incompatible libraries imported before the pinned install | Start a fresh runtime and choose **Run all** before importing extra packages. Do not bypass the notebook's version checks. |
| Model/sample digest or size verification fails | A download is incomplete or upstream bytes differ from the pinned artifact | Remove the affected runtime cache/download and rerun. Do not disable the integrity check. |
| Out-of-memory / session restart | Too many large models or intermediate objects are resident | Use the default execution tier, follow the notebook's release/unload steps, and avoid enabling optional heavy branches together. |
| BYOD validation fails | User input does not satisfy the documented schema, shape, labels, or limits | Read the validation message, correct the stated field/shape/format, and rerun the BYOD branch. |
| Your numbers differ slightly from the example/expectation | Supported hardware or library execution can introduce small numerical variation | Compare the qualitative pattern, metric definitions, split, and model revision before treating the difference as meaningful. |


# Glossary

| Term | Meaning in this notebook |
|---|---|
| **Embedding** | A numerical vector representation used here to compare semantic similarity. |
| **Bi-encoder / retriever** | Encodes queries and documents separately so document vectors can be reused. |
| **Cosine similarity** | A similarity measure based on the angle between vectors. |
| **Top-k** | The first k candidates returned by the retriever. |
| **Cross-encoder / reranker** | Scores a query and candidate jointly, using more compute for a smaller shortlist. |
| **Recall@k** | Whether the relevant item appears within the first k retrieved candidates. |